<a href="https://colab.research.google.com/github/AvanthaPeiris/n8n/blob/master/LanGraph_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain==1.5.0 -q
!pip install langchain.openai -q
!pip install langgraph -q
##!pip install -U langchain-google-genai langchain-core -q

In [2]:
!pip install -U langchain-google-genai langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 18.6 MB/s  0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [langchain-google-genai]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.63.1 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.6.1 which is incompatible.


In [52]:
from typing import TypedDict, List, Union # stacking both user and AI messages via Union
from langchain_core.messages import HumanMessage, AIMessage  # geting both user and AI messages via AIMessage

In [53]:
class agentState(TypedDict):
  msg: List[Union[HumanMessage,AIMessage]]

In [54]:
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('ChatGPT')

In [55]:
import os
import getpass
os.environ["GOOGLE_API_KEY"] = userdata.get('Gemini_new')

# #getpass.getpass("Enter your Gemini API key: ")


In [8]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [56]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0
)

In [61]:
def chatBot(state:agentState) -> agentState:
  # Using with_retry on the runnable to automatically retry on transient API errors
  runnable_with_retry = llm.with_retry(
      stop_after_attempt=3,
      wait_exponential_jitter=True
  )
  response = runnable_with_retry.invoke(state['msg'])
  #response = model.invoke(state['msg'])
  state['msg'].append(AIMessage(content=response.content)) # adding AI messages
  print(f'\nAI: {response.content}')
  return state

In [58]:
from langgraph.graph import StateGraph,START,END

workflow=StateGraph(agentState)

workflow.add_node("chatbot",chatBot)

workflow.add_edge(START,"chatbot")
workflow.add_edge("chatbot",END)

app=workflow.compile()




In [59]:
user_prompt=input('Enter your query :')
results = app.invoke({"msg":[HumanMessage(content=user_prompt)]})
print(results)

KeyboardInterrupt: Interrupted by user

In [63]:
chatHistory=[]

user_prompt=input('Enter your query :')
while user_prompt!='exit':
  chatHistory.append(HumanMessage(content=user_prompt))
  results = app.invoke({"msg":chatHistory})
  chatHistory=results['msg']
  #print(results)
  user_prompt=input('Enter your query :')
print("Bye for now! see you soon!")

Enter your query :hi good morning


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:4059: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



AI: [{'type': 'text', 'text': 'Good morning! How can I help you today?', 'extras': {'signature': 'EpYBCpMBAWkUfRPsj8gWoZEs5MahwJEw4k4TYhCawI66JzNdM8lXK/tmp36iuMGqW0empFszbQ6XxMTnSbJe6UgJA8fCeuKSEc3fMLpNy9UHNyPnh4kngSbatQhL+PFf/NL1QRqmtwvhn3CNwzO3ukp/eKE+wMEENYTsCRIiBv8+BVhH6yFD1jEFT365ekRtngt3ZSegKOAz'}}]
Enter your query :I'm avantha

AI: [{'type': 'text', 'text': "Good morning, Avantha! It's nice to meet you. How can I help you today?", 'extras': {'signature': 'EpYBCpMBAWkUfROnSgoXTHl9wtEEPDl2vYBVs1e57zrfAl59B/PGcb1l9qGxWucYffAqu6he5402X/KeMLQyYxLb4gmHTnSdBEBqWDqSDsZLGK/ZCX/PIGX6kW1VPf4BJxdUlm5IPWRfZbCEbOcvUOcbsVCBfTiJB+PPBNR7R20qaEcEjnDJmrJ2AQL2G+EX6dbc4X6J2oaJ'}}]
Enter your query :who am I i? and what half of the day we are now in?

AI: [{'type': 'text', 'text': 'Good morning, Avantha! \n\nYou are Avantha, a wonderful human being chatting with me, an AI assistant. \n\nAs for the time of day, since you mentioned it is "morning," we are currently in the **first half** of the day! (